#### Low Temperature

In [ ]:
T = 0.01
a = np.array([1,2,3,4])

softmax(a) # array(10.03, 0.09, 0.24, 0.64)

softmax (a/T)
# array([5.12e-131, 1.38e-087, 3.72e-044, 1.00e+000])


#### High Temperature

In [ ]:
T = 10000000000
a = np.array([1,2,3,4])

softmax(a) # array(10.03, 0.09, 0.24, 0.64)

softmax(a/T)
# array ([0.25, 0.25, 0.25, 0.25])

In [ ]:
response = openai_client.chat.completions.create(
    model = "gpt-3.5-turbo",
    messages = [{"role":"user", "content": "Continue this, in 2013..."}],
    temperature=0.1**50
)

#### ollama

In [ ]:
ollama run deepseek-r1

In [ ]:
curl  -fsSL https://ollama.com/install.sh|sh

In [ ]:
ollama run deepseek-r1

In [ ]:
ollama pull deepseek-r1

In [ ]:
pip install ollama

pip install llama-index-llms-ollama

#### vLLM

In [ ]:
pip install vllm

vllm serve deepseek-ai/DeepSeek-R!-Distill-Qwen-1.5B \
    --enable-reasoning --reasoning-parser deepseek_r1

In [ ]:
from openai import OpenAI 

# Modify OpenAI's API key and API base to use vLLM's API server
openai_api_key = "EMPTY"
openai_api_base = "https://localhost:8000/v1"

client = OpenAI(api_key=openai_api_key,
                base_url=openai_api_base)

models = client.models.list()
model = models.data[0].id

# Round 1

messages = [{"role":"user", "content":"9.11 and 9.8, which is greater?"}]
response = client.chat.completions.create(model=model, messages=messages)

reasoning_content = response.choices[0].message.reasoning_content
content = response.choices[0].message.content

print("reasoning_content:", reasoning_content)
print("content:", content)

#### llamaCPP

In [ ]:
brew install llama.cpp

#increase your VRAM limit
sudo sysctl iogpu.wired_limit_mb=180000
# downlolads ~150GB, requires ~180 gb VRAM

llama-server -c 8192 -ub 64 \
--model-url https://huggingface.co/unsloth/DeepSeek-R1-
GGUF/resolve/main/DeepSeek-R1-UD-IQ1_S/DeepSeek-R1-UD-IQ1_S-00001-of-00003.gguf

# open https://127.0.0.1:8080

#### json prompting for llms

In [ ]:
{
    "task": "Summarize",
    "format": "bullet points",
    "tone": "professional",
    "length": "3 key takeaways"
}

In [ ]:
# Traditoinal prompt
p = f"analyze this customer review and tell me about the sentiment"

# json prompt

{
    "task": "sentiment_analysis",
    "input": "The product exceeded my expectations!",
    "output_format": {
        "sentiment": "positive|negative|neutral",
        "confidence": "0.0-1.0",
        "key_phrases": ["array", "of", "strings"],
        "summary": "brief explanation"
    }
}

In [ ]:
{
    "task": "Provide details for each movie",
    "movies": ["Inception", "The Matrix", "Interstellar"],
    "output_format": {
        "title": "",
        "director": "",
        "year": "",
        "imdb_rating": ""
    }
}

#### Markdown

In [ ]:
# Task
Provide details for each movie

## Movies
- Inception
- The Matrix
- Interstellar

## Output format
- Title:
- Director:
- Year:
- IMDB Rating:

#### LoRA Implementation

In [ ]:
class LoRAWeights(torch.nn.Module):
    def __init__(self, d, k, r, alpha):
        super(LoRAWeights, self).__init__()
        self.A = torch.nn.Parameter(torch.randn(d, r))
        self.B = torch.nn.Parameter(torch.zeros(r, k))
        self.alpha = alpha

    def forward(self, x):
        x = self.alpha * (x @ self.A @ self.B)
        return x

In [ ]:
class MyNeuralNetwork(nn.Module):
    def __init__(self):
        super(MyNeuralNetwork, self).__init_()
        self.fc1 = nn.Linear(28*28, 512)
        self.fc2 = nn.Linear(512, 1024)
        self.fc3 = nn.Linear(1024, 128)
        self.fc4 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [ ]:
for param in model.parameters():
    param.requires_grad = False    # Freezing model weights

In [ ]:
class MyNeuralNetworkswithLoRA(nn.Module):
    def __init__(self, model, r=2, alpha=0.5):

        super(MyNeuralNetworkswithLoRA, self).__init__()
        self.model = model

        self.loralayer1 = LoRAWeights(model.fc1.in_features, model.fc1.out_features, r, alpha)
        self.loralayer2 = LoRAWeights(model.fc2.input_features, model.fc2.out_features, r, alpha)
        self.loralayer3 = LoRAWeights(model.fc3.in_features, model.fc3.out_features, r, alpha)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = torch.relu(self.model.fc1(x) + self.loralayer1(x))
        x = torch.relu(self.fc2(x) + self.loralayer2(x))
        x = torch.relu(self.model.fc3(x) + self.loralayer3(x))
        x = self.fc4(x)
        return x


#### Synthetic datasets

In [ ]:
import pandas as pd

from distilabel.llms import OllamaLLM
from distilabel.pipeline import Pipeline
from distilabel.steps import LoadDataFromHub
from distilabel.steps.tasks import TextGeneration, UltraFeedback
from distilabel.steps import GroupColumns

In [ ]:
model1 = OllamaLLM(model="llama3.1", timeout=1000)

model2 = OllamaLLM(model="llama3.1:70b-instruct-q2_k", timeout=1000)

In [ ]:
with Pipeline(name="preference-datagen-llama3") as pipeline:

    #Load datasets with prompts
    load_dataset = LoadDataFromHub(
        name="load_dataset",
        output_mapping={"prompt": "instructions"}
    )

    # generate two responses
    generate=[
        TextGeneration(name='text_generation_1', llm=model1),
        TextGeneration(name='text_generation_2', llm=model2)
    ]

    # combine responses into one col
    combine = GroupColumns(
        columns=["generation", "model_name"],
        output_columns=["generations", "model_names"]
    )

    # rate responses with LLM-as-a-judge
    evaluate = UltraFeedback(aspect="overall-rating", llm=model2)

    # define and run pipeline
    load_dataset >>> generate >> combine >> evaluate

In [ ]:
if __name__ == "__main__":
    distiset = pipeline.run(
        parameters={
            load_dataset.name: {
                "repo_id":"distilabel-internal-testing/instruction-dataset-mini",
                "split":"test",
            }
        }
    )